In [ ]:
import pandas as pd
import numpy as np
import sqlite3

sales_raw = pd.read_csv("data/sales_train_evaluation.csv")
calendar = pd.read_csv("data/calendar.csv")
prices = pd.read_csv("data/sell_prices.csv")

STORES = ["CA_1", "CA_2", "CA_3"]
day_cols = [c for c in sales_raw.columns if c.startswith("d_")]

# top 100 items by total CA sales
ca = sales_raw[sales_raw["store_id"].isin(STORES)].copy()
ca["total"] = ca[day_cols].sum(axis=1)
top_items = (ca.groupby("item_id")["total"].sum()
               .nlargest(100).index.tolist())
sub = ca[ca["item_id"].isin(top_items)]

# wide -> long
long = sub.melt(id_vars=["item_id", "store_id"], value_vars=day_cols,
                var_name="d", value_name="units")
long = long.merge(calendar[["d", "date"]], on="d")
print("sales rows:", len(long))   # expect ~580k

# write the three tables
con = sqlite3.connect("forecast.db")
long[["item_id", "store_id", "date", "units"]].to_sql(
    "sales", con, if_exists="replace", index=False)
calendar[["d", "date", "wm_yr_wk", "weekday", "event_name_1", "snap_CA"]].to_sql(
    "calendar", con, if_exists="replace", index=False)
prices[prices["store_id"].isin(STORES) & prices["item_id"].isin(top_items)].to_sql(
    "prices", con, if_exists="replace", index=False)

# indexes: what makes per-product queries fast
con.execute("CREATE INDEX IF NOT EXISTS idx_sales ON sales(item_id, store_id, date)")
con.execute("CREATE INDEX IF NOT EXISTS idx_prices ON prices(item_id, store_id, wm_yr_wk)")
con.commit()
print("db built")

In [ ]:
FEATURE_SQL = """
WITH daily AS (
    SELECT s.item_id, s.store_id, s.date, s.units,
           c.event_name_1, c.snap_CA, c.wm_yr_wk
    FROM sales s
    JOIN calendar c ON s.date = c.date
)
SELECT d.item_id, d.store_id, d.date, d.units,
       CAST(strftime('%w', d.date) AS INT)            AS dayofweek,
       CAST(strftime('%m', d.date) AS INT)            AS month,
       CASE WHEN d.event_name_1 IS NOT NULL THEN 1 ELSE 0 END AS is_event,
       d.snap_CA,
       LAG(d.units, 7)  OVER w                        AS lag_7,
       LAG(d.units, 14) OVER w                        AS lag_14,
       LAG(d.units, 28) OVER w                        AS lag_28,
       AVG(d.units) OVER (w ROWS BETWEEN 7 PRECEDING AND 1 PRECEDING)  AS roll_mean_7,
       AVG(d.units) OVER (w ROWS BETWEEN 28 PRECEDING AND 1 PRECEDING) AS roll_mean_28,
       p.sell_price
FROM daily d
LEFT JOIN prices p
  ON p.item_id = d.item_id AND p.store_id = d.store_id AND p.wm_yr_wk = d.wm_yr_wk
WINDOW w AS (PARTITION BY d.item_id, d.store_id ORDER BY d.date)
ORDER BY d.item_id, d.store_id, d.date
"""

feat_sql = pd.read_sql(FEATURE_SQL, con, parse_dates=["date"])
print(feat_sql.shape)
feat_sql.head()

In [ ]:
feat2 = pd.read_csv("app_data.csv", parse_dates=["date"])

In [ ]:
chk = feat_sql[(feat_sql["item_id"] == "FOODS_3_090") & (feat_sql["store_id"] == "CA_1")].tail(5)
old = feat2[feat2["item_id"] == "FOODS_3_090"].tail(5)
print(chk[["date", "units", "lag_7", "roll_mean_7"]])
print(old[["date", "units", "lag_7", "roll_mean_7"]])

In [ ]:
import os
print(f"forecast.db: {os.path.getsize('forecast.db') / 1e6:.1f} MB")